<a href="https://colab.research.google.com/github/YUGOROU/amd-voice-sft/blob/feature%2Fdata-preprocessing/preprocess.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AMD Voice SFT — Data Preprocessing

Converts 3 HuggingFace datasets to ChatML format for SFT training.

**Datasets:**
- `fadodr/mental_health_therapy` (8,580 train rows, single-turn)
- `Estwld/empathetic_dialogues_llm` (19,500 train rows, multi-turn)
- `HuggingFaceTB/everyday-conversations-llama3.1-2k` (2,263 train rows, multi-turn)

**Output:** `data/train.jsonl`, `data/val.jsonl` (90:10 split)

In [ ]:
!uv pip install -q datasets tqdm huggingface_hub

In [ ]:
import json
import os
import random
from datasets import load_dataset
from tqdm import tqdm

SYSTEM_PROMPT = (
    "You are a caring and patient AI companion for elderly users with "
    "memory difficulties. Always respond with warmth, patience, and clarity."
)

SEED = 42
VAL_RATIO = 0.1

In [ ]:
# --- Conversion functions ---

def convert_mental_health(ex):
    """fadodr/mental_health_therapy: instruction/input/output -> ChatML"""
    user_text = ex["input"].strip()
    assistant_text = ex["output"].strip()
    if not user_text or not assistant_text:
        return None
    return {"messages": [
        {"role": "system",    "content": SYSTEM_PROMPT},
        {"role": "user",      "content": user_text},
        {"role": "assistant", "content": assistant_text},
    ]}


def convert_empathetic(ex):
    """Estwld/empathetic_dialogues_llm: conversations (role/content list) -> ChatML"""
    convs = ex["conversations"]
    if not convs or convs[0]["role"] != "user":
        return None
    return {"messages": [{"role": "system", "content": SYSTEM_PROMPT}] + convs}


def convert_everyday(ex):
    """HuggingFaceTB/everyday-conversations: messages (role/content list) -> ChatML"""
    msgs = ex["messages"]
    if not msgs or msgs[0]["role"] != "user":
        return None
    return {"messages": [{"role": "system", "content": SYSTEM_PROMPT}] + msgs}

In [ ]:
# --- Load and convert ---

datasets_config = [
    ("fadodr/mental_health_therapy",                    "train", convert_mental_health),
    ("Estwld/empathetic_dialogues_llm",                 "train", convert_empathetic),
    ("HuggingFaceTB/everyday-conversations-llama3.1-2k","train", convert_everyday),
]

all_samples = []
source_counts = {}

for ds_id, split, fn in datasets_config:
    print(f"Loading {ds_id} [{split}]...")
    ds = load_dataset(ds_id, split=split)
    converted = [fn(ex) for ex in tqdm(ds, desc=ds_id.split("/")[1])]
    valid = [s for s in converted if s is not None]
    source_counts[ds_id] = {"raw": len(ds), "converted": len(valid)}
    all_samples.extend(valid)
    print(f"  {len(ds)} -> {len(valid)} samples")

print(f"\nTotal before filtering: {len(all_samples)}")

In [ ]:
# --- Quality filter ---

def is_quality(sample):
    for m in sample["messages"]:
        if m["role"] == "user"      and len(m["content"]) < 20:
            return False
        if m["role"] == "assistant" and len(m["content"]) < 50:
            return False
    return True

before = len(all_samples)
all_samples = [s for s in all_samples if is_quality(s)]
print(f"Quality filter: {before} -> {len(all_samples)} ({len(all_samples)/before*100:.1f}% kept)")

In [ ]:
# --- Shuffle and split 90:10 ---

random.seed(SEED)
random.shuffle(all_samples)

split_idx = int(len(all_samples) * (1 - VAL_RATIO))
train_data = all_samples[:split_idx]
val_data   = all_samples[split_idx:]

print(f"Train: {len(train_data):,}  |  Val: {len(val_data):,}")

In [ ]:
# --- Save as JSONL ---

os.makedirs("data", exist_ok=True)

for name, data in [("train", train_data), ("val", val_data)]:
    path = f"data/{name}.jsonl"
    with open(path, "w", encoding="utf-8") as f:
        for s in data:
            f.write(json.dumps(s, ensure_ascii=False) + "\n")
    print(f"Saved {path} ({len(data):,} samples)")

# Spot-check first sample
print("\n--- Sample train[0] ---")
print(json.dumps(train_data[0], indent=2, ensure_ascii=False))

In [ ]:
# --- Upload to Hugging Face Hub ---
# Uses HF_TOKEN from Colab Secrets (left panel > key icon > add HF_TOKEN)

from google.colab import userdata
from huggingface_hub import HfApi

hf_token = userdata.get("HF_TOKEN")
api = HfApi(token=hf_token)

api.create_repo(
    repo_id="YUGOROU/amd-voice-sft-data",
    repo_type="dataset",
    private=True,
    exist_ok=True,
)
api.upload_folder(
    folder_path="data",
    repo_id="YUGOROU/amd-voice-sft-data",
    repo_type="dataset",
)
print("Uploaded to HF Hub: YUGOROU/amd-voice-sft-data")